# Experiment 5.3.2.3 — FF-SNN before RSNN64

Analysis-only notebook for the finalized pre-recurrent transformation ablation.

Sections: Reference contract, Primary architecture comparison, Paired deltas vs direct, Spiking-transform attribution, FF-width sensitivity, Generalization, History contribution, SNN-native accessibility, Efficiency/state dimension, and decision table.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not locate writingRing repository root")


repo_root = find_repo_root()
artifact_root = repo_root / "notebooks" / "artifacts" / "experiment_5_3_2_3_ff_before_rsnn" / "ff_before_rsnn_v1"
manifest = json.loads((artifact_root / "manifest.json").read_text(encoding="utf-8"))
runs = pd.read_csv(artifact_root / "runs.csv")
histories = pd.read_csv(artifact_root / "histories.csv")
probe_runs = pd.read_csv(artifact_root / "probe_runs.csv")
ablation_runs = pd.read_csv(artifact_root / "ablation_runs.csv")
history_gain_runs = pd.read_csv(artifact_root / "history_gain_runs.csv")
activity_runs = pd.read_csv(artifact_root / "activity_runs.csv")
representation_runs = pd.read_csv(artifact_root / "representation_runs.csv")
baseline_runs = pd.read_csv(artifact_root / "baseline_runs.csv")
local_reference = pd.read_csv(artifact_root / "local_reference.csv")
paired_deltas = pd.read_csv(artifact_root / "paired_deltas.csv")

assert manifest["expected_train_runs"] == 15
assert manifest["expected_comparison_rows"] == 20
assert len(runs) == 20
assert runs.groupby("condition")["seed"].nunique().eq(5).all()
manifest


## 1. Reference contract

The direct condition is reused Exp5.3.2.2 `rsnn_h64`. The three new conditions keep RSNN width fixed at H=64. `linear64_rsnn64` is the parameterization/depth control for `ffsnn64_rsnn64`.


In [ ]:
condition_order = [
    "direct_rsnn64",
    "linear64_rsnn64",
    "ffsnn64_rsnn64",
    "ffsnn128_rsnn64",
]
summary = runs.groupby("condition", as_index=False).agg(
    phase_ba_mean=("probe_u_test_phase_ba", "mean"),
    phase_ba_sd=("probe_u_test_phase_ba", "std"),
    progress_mae_mean=("probe_test_sample_balanced_mae_mean", "mean"),
    progress_mae_sd=("probe_test_sample_balanced_mae_mean", "std"),
    spearman_mean=("probe_test_spearman_mean", "mean"),
    violation_mean=("probe_test_monotonic_violation_rate_mean", "mean"),
    parameter_count=("parameter_count", "mean"),
)
summary["condition"] = pd.Categorical(summary["condition"], categories=condition_order, ordered=True)
summary = summary.sort_values("condition").reset_index(drop=True)
summary


## 2. Primary architecture comparison

The primary scientific test is whether `ffsnn64_rsnn64` improves the final RSNN membrane representation relative to `direct_rsnn64`.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.errorbar(summary["condition"].astype(str), summary["phase_ba_mean"], yerr=summary["phase_ba_sd"], marker="o", capsize=3)
ax.set_ylabel("Test membrane Phase BA")
ax.set_xlabel("Architecture")
ax.tick_params(axis="x", rotation=20)
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.errorbar(summary["condition"].astype(str), summary["progress_mae_mean"], yerr=summary["progress_mae_sd"], marker="o", capsize=3)
ax.set_ylabel("Test sample-balanced Progress MAE")
ax.set_xlabel("Architecture")
ax.tick_params(axis="x", rotation=20)
ax.grid(alpha=0.25)
plt.show()


## 3. Paired deltas vs direct RSNN64

Positive `delta_phase_ba` is better. Negative `delta_progress_mae` and negative `delta_violation` are better. The table also reports per-seed win counts to distinguish a stable gain from a mean driven by one seed.


In [ ]:
paired_summary = paired_deltas.groupby("condition", as_index=False).agg(
    mean_delta_phase_ba=("delta_phase_ba", "mean"),
    sd_delta_phase_ba=("delta_phase_ba", "std"),
    mean_delta_progress_mae=("delta_progress_mae", "mean"),
    sd_delta_progress_mae=("delta_progress_mae", "std"),
    mean_delta_spearman=("delta_spearman", "mean"),
    mean_delta_violation=("delta_violation", "mean"),
)
win_counts = paired_deltas.groupby("condition").agg(
    phase_wins=("delta_phase_ba", lambda s: int((s > 0).sum())),
    progress_wins=("delta_progress_mae", lambda s: int((s < 0).sum())),
    joint_wins=("seed", lambda s: 0),
).reset_index()
for idx, row in win_counts.iterrows():
    condition = row["condition"]
    subset = paired_deltas[paired_deltas["condition"] == condition]
    win_counts.loc[idx, "joint_wins"] = int(((subset["delta_phase_ba"] > 0) & (subset["delta_progress_mae"] < 0)).sum())
paired_summary.merge(win_counts, on="condition")


## 4. Spiking-transform attribution

`linear64_rsnn64` and `ffsnn64_rsnn64` have matched weight shapes and paired initialization. Their difference isolates the short-tau LIF/spike transform much more cleanly than comparing either one only against the direct network.


In [ ]:
pair = runs[runs["condition"].isin(["linear64_rsnn64", "ffsnn64_rsnn64"])].pivot(index="seed", columns="condition")
mechanism = pd.DataFrame({
    "seed": pair.index,
    "delta_phase_ba_ff_minus_linear": pair["probe_u_test_phase_ba"]["ffsnn64_rsnn64"] - pair["probe_u_test_phase_ba"]["linear64_rsnn64"],
    "delta_progress_mae_ff_minus_linear": pair["probe_test_sample_balanced_mae_mean"]["ffsnn64_rsnn64"] - pair["probe_test_sample_balanced_mae_mean"]["linear64_rsnn64"],
    "delta_spearman_ff_minus_linear": pair["probe_test_spearman_mean"]["ffsnn64_rsnn64"] - pair["probe_test_spearman_mean"]["linear64_rsnn64"],
})
mechanism


## 5. FF-width sensitivity

Compare `ffsnn128_rsnn64` with `ffsnn64_rsnn64`. The recurrent state is fixed, so a consistent gain from FF128 indicates that the front-end transformation width, not recurrent width, remains capacity-limited.


In [ ]:
ff_pair = runs[runs["condition"].isin(["ffsnn64_rsnn64", "ffsnn128_rsnn64"])].pivot(index="seed", columns="condition")
ff_width_delta = pd.DataFrame({
    "seed": ff_pair.index,
    "delta_phase_ba_128_minus_64": ff_pair["probe_u_test_phase_ba"]["ffsnn128_rsnn64"] - ff_pair["probe_u_test_phase_ba"]["ffsnn64_rsnn64"],
    "delta_progress_mae_128_minus_64": ff_pair["probe_test_sample_balanced_mae_mean"]["ffsnn128_rsnn64"] - ff_pair["probe_test_sample_balanced_mae_mean"]["ffsnn64_rsnn64"],
})
ff_width_delta


## 6. Generalization

A transformed model should not be called better merely because its train representation improves. Compare train/validation/test probe performance and the existing train-val gaps.


In [ ]:
generalization = runs.groupby("condition", as_index=False).agg(
    train_phase=("probe_u_train_phase_ba", "mean"),
    val_phase=("probe_u_val_phase_ba", "mean"),
    test_phase=("probe_u_test_phase_ba", "mean"),
    train_mae=("probe_train_sample_balanced_mae_mean", "mean"),
    val_mae=("probe_val_sample_balanced_mae_mean", "mean"),
    test_mae=("probe_test_sample_balanced_mae_mean", "mean"),
    phase_gap=("probe_phase_train_val_gap", "mean"),
    progress_gap=("probe_progress_val_train_gap", "mean"),
)
generalization


## 7. History contribution

`H_reset` measures performance lost when temporal state is reset every timestep. `H_shuffle` measures performance lost when within-gesture temporal order is destroyed. For FF-SNN conditions, state reset removes both FF and recurrent state.


In [ ]:
history_summary = history_gain_runs.groupby("condition", as_index=False).agg(
    H_reset_phase=("H_reset_phase_ba", "mean"),
    H_shuffle_phase=("H_shuffle_phase_ba", "mean"),
    H_reset_progress=("H_reset_progress_mae", "mean"),
    H_shuffle_progress=("H_shuffle_progress_mae", "mean"),
    H_reset_spearman=("H_reset_spearman", "mean"),
    H_shuffle_spearman=("H_shuffle_spearman", "mean"),
)
history_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.plot(history_summary["condition"], history_summary["H_reset_phase"], marker="o", label="H_reset phase")
ax.plot(history_summary["condition"], history_summary["H_shuffle_phase"], marker="o", label="H_shuffle phase")
ax.axhline(0.0, linewidth=1)
ax.set_ylabel("Phase BA history gain")
ax.tick_params(axis="x", rotation=20)
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 8. SNN-native accessibility

Compare membrane, synaptic, instantaneous spike, trailing-250ms count, and trailing-500ms count. A useful FF stage should ideally not improve membrane WHEN while making the information much harder to access from spikes.


In [ ]:
accessibility = probe_runs.groupby(["condition", "feature_type"], as_index=False).agg(
    phase_ba=("phase_probe_test_ba", "mean"),
    progress_mae=("progress_probe_test_sample_balanced_mae_mean", "mean"),
)
accessibility


## 9. Efficiency and effective state dimension

All conditions end in RSNN64, so compare parameter count, recurrent firing utilization, `effective_dimension`, and `pca90_dimension`.


In [ ]:
activity_summary = activity_runs.groupby("condition", as_index=False).agg(
    mean_fr=("mean_firing_rate", "mean"),
    median_neuron_fr=("median_neuron_firing_rate", "mean"),
    dead_fraction=("dead_neuron_fraction", "mean"),
    highly_active_fraction=("highly_active_neuron_fraction", "mean"),
)
dimension_summary = representation_runs.groupby("condition", as_index=False).agg(
    effective_dimension=("effective_dimension", "mean"),
    pca90_dimension=("pca90_dimension", "mean"),
    total_variance=("total_variance", "mean"),
)
efficiency = summary[["condition", "parameter_count"]].merge(activity_summary, on="condition").merge(dimension_summary, on="condition")
efficiency


## 10. Decision table

There is no scalar composite score. First identify strict mean Pareto improvements over direct RSNN64: higher Phase BA and lower Progress MAE. Then inspect paired consistency, history gains, generalization, spike accessibility, and cost.


In [ ]:
direct_row = summary[summary["condition"].astype(str) == "direct_rsnn64"].iloc[0]
decision = summary.copy()
decision["beats_direct_phase"] = decision["phase_ba_mean"] > direct_row["phase_ba_mean"]
decision["beats_direct_progress"] = decision["progress_mae_mean"] < direct_row["progress_mae_mean"]
decision["strict_pareto_vs_direct"] = decision["beats_direct_phase"] & decision["beats_direct_progress"]
decision.merge(history_summary, on="condition", how="left").merge(activity_summary, on="condition", how="left").merge(dimension_summary, on="condition", how="left")
